In [1]:
!pip install --break-system-packages clickhouse-connect

  Using cached clickhouse_connect-0.10.0-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (4.2 kB)
  Using cached zstandard-0.25.0-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (3.3 kB)
  Using cached lz4-4.4.5-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (3.8 kB)
Using cached clickhouse_connect-0.10.0-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (1.1 MB)
Using cached lz4-4.4.5-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (1.4 MB)
Using cached zstandard-0.25.0-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (5.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [clickhouse-connect]


In [2]:
import clickhouse_connect
import pandas as pd
import os

pd.options.display.max_colwidth = 100

database = 'snccnmb'

client = clickhouse_connect.get_client(host='clickhouse01', port=8123, username=os.getenv('CLICKHOUSE_USER'), password=os.getenv('CLICKHOUSE_PASSWORD'))

In [3]:
client.command(f'''
    CREATE DATABASE IF NOT EXISTS {database} ON CLUSTER '{{cluster}}';
''')

['clickhouse01',
 '9000',
 '0',
 '',
 '3',
 '0\nclickhouse02',
 '9000',
 '0',
 '',
 '2',
 '0\nclickhouse04',
 '9000',
 '0',
 '',
 '1',
 '0\nclickhouse03',
 '9000',
 '0',
 '',
 '0',
 '0']

# 1 ↓

In [ ]:
client.command(f'''
	DROP TABLE IF EXISTS {database}.orders_log;
''')

client.command(f'''
	CREATE TABLE {database}.orders_log
(
    order_id        UInt32,
    user_id  	 	UInt64,
    order_date      Date,
    total_amount	Decimal(9, 1),
	paid 			Enum('оплачено' = 1,
						 'не оплачено' = 0),
	items 		 	Array(UInt32)
)
    ENGINE = Log;
''')

client.command(f'''
	INSERT INTO {database}.orders_log VALUES
	(1, 123, '2026-02-01', 34, 1, [23, 33, 86, 4]),
	(2, 26, '2026-01-01', 23647.36, 0, [234, 64]),
	(3, 75, '2026-01-19', 10.1, 1, [1, 2, 3])
''')

client.query_df(f'''
	SELECT * FROM {database}.orders_log
''')

,order_id,user_id,order_date,total_amount,paid,items
0,1,123,2026-02-01,34.00,оплачено,"[23, 33, 86, 4]"
1,2,26,2026-01-01,23647.36,не оплачено,"[234, 64]"
2,3,75,2026-01-19,10.10,оплачено,"[1, 2, 3]"


# 2 ↓

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.table_1;
''')

client.command(f'''
CREATE TABLE  {database}.table_1
(
    order_id     UInt32, 
    user_id      UInt16,  
    total_amount Float32,   
    paid         Nullable(UInt8),     
    item_count   Int8   
)
ENGINE = Log;
''')

client.command(f'''
    INSERT INTO {database}.table_1 VALUES (10001, 321, 1499.50, 1, 3), (10001,  321, 1499.50 , NULL, 3), (10001, 321, , 1, 3);
''')

client.command(f'''
    DROP TABLE IF EXISTS {database}.table_2;
''')

client.command(f'''
CREATE TABLE {database}.table_2
(
    order_id     Int64, 
    user_id      UInt32,  
    total_amount Nullable(Decimal(5, 2)),   
    paid         UInt8,     
    item_count   String   
)
ENGINE = Log;
''')

client.command(f'''
    INSERT INTO {database}.table_2 VALUES (10001, 123456, 899.99, 0, '5'), (10001, 123456, NULL, 0, '5'), (10001, 123456, 899.99, 0, '5 тут специально вписаны буквы');
''')

In [20]:
client.query_df(f'''
    SELECT order_id, user_id, total_amount, paid, item_count FROM {database}.table_1
    UNION ALL
    SELECT order_id, user_id, toFloat32(total_amount), paid, toInt8OrNull(item_count) FROM {database}.table_2
''')


,order_id,user_id,total_amount,paid,item_count
0,10001,321,1499.50000,1,3
1,10001,321,1499.50000,<NA>,3
2,10001,321,0.00000,1,3
3,10001,123456,899.98999,0,5
4,10001,123456,NaN,0,5
5,10001,123456,899.98999,0,<NA>


# 3 ↓

In [23]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.ttl_test;
''')

client.command(f'''
CREATE TABLE  {database}.ttl_test
(
    id	UInt32, 
    dt	Datetime  
)
ENGINE = MergeTree
ORDER BY(id)
TTL dt + INTERVAL 10 SECOND DELETE;
''')

In [46]:
client.command(f'''
    INSERT INTO {database}.ttl_test
	SELECT number, now() + number FROM numbers(100);
''')

In [77]:
client.query_df(f'''
	SELECT *
	FROM {database}.ttl_test
''')

""


In [76]:
client.command(f'''
    OPTIMIZE TABLE {database}.ttl_test FINAL;
''')

# 4 ↓

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.ttl_attributes;
''')

client.command(f'''
CREATE TABLE  {database}.ttl_attributes
(
    sale_id UInt64 COMMENT 'Уникальный идентификатор продажи', 
    price Float32 DEFAULT 0.0,
	quantity UInt8 DEFAULT 1,
	total Float32 MATERIALIZED price * quantity,
	description String CODEC(ZSTD(1)),
	taxed_total Float32 ALIAS total * 1.2
)
ENGINE = MergeTree
ORDER BY(sale_id);
''')

In [91]:
client.command(f'''
    INSERT INTO {database}.ttl_attributes (
	sale_id,
	price,
	quantity,
	description
	) VALUES (1, 22.33, 100000, 'DADAAFAFEA');
''')

In [ ]:
client.command(f'''
    INSERT INTO {database}.ttl_attributes (
	total,
	taxed_total
	) VALUES (23.215234, 745.333);
''')

In [ ]:
client.query_df(f'''
	SELECT 
		sale_id,
		price,
		quantity,
		total,
		description,
		taxed_total
	FROM {database}.ttl_attributes
''')

,sale_id,price,quantity,total,description,taxed_total
0,1,22.33,160,3572.800049,DADAAFAFEA,4287.359863


# 5 ↓

In [104]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.partition_test;
''')

client.command(f'''
CREATE TABLE  {database}.partition_test
(
    order_id UInt64, 
    order_date Datetime,
	customer_id UInt32,
	product_name String,
	quantity UInt8,
	price Float32,
	total Float32 MATERIALIZED quantity * price
)
ENGINE = MergeTree
PARTITION BY toYYYYMM(order_date)
ORDER BY(customer_id, order_date);
''')

In [106]:
client.command(f'''
    INSERT INTO {database}.partition_test (
	order_id, 
    order_date,
	customer_id,
	product_name,
	quantity,
	price
	) VALUES 
	(1, now() - INTERVAL 60 DAY, 234, 'COCK', 2, 200000000),
	(2, now() - INTERVAL 30 DAY, 97, 'CHILD', 1, 12.123),
	(3, now(), 221, 'apple', 1, 4.31);
''')

In [107]:
client.query_df(f'''
	SELECT 
		_part, *
	FROM {database}.partition_test
''')

,_part,order_id,order_date,customer_id,product_name,quantity,price
0,202512_1_1_0,1,2025-12-04 18:21:30+03:00,234,COCK,2,2.000000e+08
1,202602_3_3_0,3,2026-02-02 18:21:30+03:00,221,apple,1,4.310000e+00
2,202601_2_2_0,2,2026-01-03 18:21:30+03:00,97,CHILD,1,1.212300e+01


# 6 ↓

In [5]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.page_views;
''')

client.command(f'''
    CREATE TABLE {database}.page_views
    (
        user_id UInt32,
        page String,
        device String,
        view_time UInt8
    )
    ENGINE = MergeTree
    ORDER BY user_id;
''')

client.command(f'''
    INSERT INTO {database}.page_views (user_id, page, device, view_time) VALUES
        (1, 'home', 'mobile', 10),
        (1, 'catalog', 'mobile', 15),
        (2, 'home', 'web', 8),
        (2, 'checkout', 'web', 20),
        (2, 'home', 'mobile', 5),
        (4, 'catalog', 'web', 12),
        (5, 'checkout', 'mobile', 7),
        (6, 'home', 'web', 9),
        (7, 'home', 'mobile', 11),
        (8, 'catalog', 'web', 13);
''')

In [8]:
client.query_df(f'''
	SELECT 
		*
	FROM {database}.page_views
''')

,user_id,page,device,view_time
0,1,home,mobile,10
1,1,catalog,mobile,15
2,2,home,web,8
3,2,checkout,web,20
4,2,home,mobile,5
5,4,catalog,web,12
6,5,checkout,mobile,7
7,6,home,web,9
8,7,home,mobile,11
9,8,catalog,web,13


### 1 Запрос ↓

In [15]:
client.query_df(f'''
	SELECT uniqIf(user_id, device = 'mobile') as unique_count
	FROM {database}.page_views
''')

,unique_count
0,4


### 2 Запрос ↓

In [ ]:
client.query_df(f'''
	SELECT avgIf(view_time, page = 'home') as avg_view_time
	FROM {database}.page_views
''')

,"avgIf(view_time, equals(page, 'home'))"
0,8.6


### 3 Запрос ↓

In [16]:
client.query_df(f'''
	SELECT device, uniq(user_id) as unique_count
	FROM {database}.page_views
	GROUP BY device
''')

,device,unique_count
0,web,4
1,mobile,4


### 4 Запрос ↓

In [6]:
client.query_df(f'''
	SELECT groupArrayIf(page, user_id = 2 AND device = 'web') as page_list
	FROM {database}.page_views
''')

,page_list
0,"[home, checkout]"


# 7 ↓

In [26]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.summing_mt_dz;
''')

client.command(f'''
    CREATE TABLE {database}.summing_mt_dz
    (
        id UInt32,
        val SimpleAggregateFunction(sum, UInt64),
        dt SimpleAggregateFunction(max, Datetime),
        example UInt32
    )
    ENGINE = AggregatingMergeTree 
    ORDER BY id
	PARTITION BY toYYYYMM(dt);
''')

client.command(f'''
    INSERT INTO {database}.summing_mt_dz
    SELECT 
        number % 2, 
        (number + 1) * 10, 
        now() + number * 60 * 60 * 24,
        (number + 1) * 100 
    from numbers(30);
''')

In [27]:
client.query_df(f'''
	SELECT *
	FROM {database}.summing_mt_dz
''')

,id,val,dt,example
0,0,560,2026-03-04 13:06:39+03:00,2700
1,1,840,2026-03-05 13:06:39+03:00,2600
2,0,1690,2026-02-28 13:06:39+03:00,100
3,1,1560,2026-02-27 13:06:39+03:00,200


# 8 ↓

In [28]:
import psycopg2 as ps
import pandas as pd
import os

schema = 'chatov' # В рамках схемы задайте свою фамилию

conn = ps.connect(host="postgres_source", 
                  port = 5432, 
                  database="dev", 
                  user=os.getenv("POSTGRES_USER"), 
                  password=os.getenv("POSTGRES_PASSWORD"))

cursor = conn.cursor()

cursor.execute(f'''
    CREATE SCHEMA IF NOT EXISTS {schema};
    ''')
    
cursor.execute(f'''
    DROP TABLE IF EXISTS {schema}.departments CASCADE;
''')

cursor.execute(f'''
    CREATE TABLE {schema}.departments (
        dept_id SERIAL PRIMARY KEY,
        dept_name VARCHAR(50),
        location VARCHAR(50)
    )
''')

cursor.execute(f'''
    INSERT INTO {schema}.departments (dept_name, location) VALUES
    ('HR', 'Москва'),
    ('IT', 'Санкт-Петербург'),
    ('Finance', 'Москва'),
    ('DE', 'Краснодар')
''')

conn.commit()

In [31]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.postgresql_table;
''')

client.command(f'''
    CREATE TABLE {database}.postgresql_table
    (
        dept_id Int32,
        dept_name String,
        location String
    )
    ENGINE = PostgreSQL(
                        'postgres_source:5432', -- сервер, порт
                        'dev',              -- БД 
                        'departments',      -- таблица
                        '{os.getenv("POSTGRES_USER")}',         -- логин
                        '{os.getenv("POSTGRES_PASSWORD")}',         -- пароль
                        'chatov'         -- имя схемы
                        );
''')

In [34]:
client.query_df(f'''
     SELECT * FROM {database}.postgresql_table
''')

,dept_id,dept_name,location
0,1,HR,Москва
1,2,IT,Санкт-Петербург
2,3,Finance,Москва
3,4,DE,Краснодар


# Подзадача 1 JIRA: из PostgreSQL  ↓

### Загрузка данных API в PostgreSQL ↓

In [ ]:
import psycopg2 as ps
import pandas as pd
import os

schema = 'chatov'

conn = ps.connect(host="postgres_source", 
                  port = 5432, 
                  database="dev", 
                  user=os.getenv("POSTGRES_USER"), 
                  password=os.getenv("POSTGRES_PASSWORD"))

cur = conn.cursor()

cur.execute(f'''
    CREATE SCHEMA IF NOT EXISTS {schema};
    ''')
    
cur.execute(f'''
    DROP TABLE IF EXISTS {schema}.YM_postgres_local;
''')

cur.execute(f'''
    CREATE TABLE {schema}.YM_postgres_local
(
    date         		 DATE,
    gender  	 		 VARCHAR(16),
    visits       		 DECIMAL(7, 1),
    users		 		 DECIMAL(7, 1),
	avgDaysBetweenVisits DECIMAL(7, 2),
	robotVisits 		 DECIMAL(7, 1),
	blockedPercentage 	 DECIMAL(7, 2),
    updated_at   		 TIMESTAMP
)
''')

with open(f'api_{database}.csv', 'r', encoding='utf-8') as f:
	cur.copy_expert(
		"""
		COPY chatov.YM_postgres_local
		FROM STDIN
		WITH CSV HEADER
		""",
		f
	)

conn.commit()

### Создаю таблички в CH ↓

In [27]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.YM_local ON CLUSTER '{{cluster}}' SYNC;
''')

client.command(f'''
CREATE TABLE {database}.YM_local ON CLUSTER '{{cluster}}'
(
    date         		 Date,
    gender  	 		 String,
    visits       		 Decimal(7, 1),
    users		 		 Decimal(7, 1),
	avgdaysbetweenvisits Decimal(7, 1),
	robotvisits 		 Decimal(7, 1),
	blockedpercentage 	 Decimal(7, 1),
    updated_at   		 DateTime64(0)
)
    ENGINE = ReplicatedReplacingMergeTree('/clickhouse/tables/{{cluster}}/{{shard}}/events_snccnmb', '{{replica}}', updated_at)
		PARTITION BY toYYYYMM(date)
        ORDER BY (date, gender)
        SETTINGS index_granularity = 8192;
''')

['clickhouse01',
 '9000',
 '0',
 '',
 '3',
 '0\nclickhouse02',
 '9000',
 '0',
 '',
 '2',
 '0\nclickhouse04',
 '9000',
 '0',
 '',
 '1',
 '0\nclickhouse03',
 '9000',
 '0',
 '',
 '0',
 '0']

In [28]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.YM_dist ON CLUSTER company_cluster SYNC;
''')

client.command(f'''
CREATE TABLE {database}.YM_dist
(
    date         		 Date,
    gender  	 		 String,
    visits       		 Decimal(7, 1),
    users		 		 Decimal(7, 1),
	avgdaysbetweenvisits Decimal(7, 1),
	robotvisits 		 Decimal(7, 1),
	blockedpercentage 	 Decimal(7, 1),
    updated_at   		 DateTime64(0)
)
    ENGINE = Distributed(
		'{{cluster}}',
		{database},
		YM_local,
		halfMD5(date)
	);
''')

### Закидываю данные из Postgres в CH ↓

In [9]:
import pandas as pd


df = pd.read_sql(
    f"SELECT * FROM {schema}.ym_postgres_local;",
    conn
)

client.insert_df(f'{database}.YM_postgres_dist', df)

# print(df.to_markdown())

/tmp/ipykernel_200/4080594586.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


In [39]:
import pandas as pd


pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

df = client.query_df(f'''
    SELECT *
    FROM {database}.YM_dist
''')
print(df.to_markdown())

|    | date                | gender        |   visits |   users |   avgdaysbetweenvisits |   robotvisits |   blockedpercentage | updated_at                |
|---:|:--------------------|:--------------|---------:|--------:|-----------------------:|--------------:|--------------------:|:--------------------------|
|  0 | 2026-01-10 00:00:00 | Not specified |       68 |      45 |                    2.3 |             0 |                27.9 | 2026-02-05 18:11:24+03:00 |
|  1 | 2026-01-10 00:00:00 | female        |       41 |      27 |                    2.9 |             1 |                30.7 | 2026-02-05 18:11:24+03:00 |
|  2 | 2026-01-10 00:00:00 | male          |      215 |     125 |                    2.3 |             1 |                47.6 | 2026-02-05 18:11:24+03:00 |
|  3 | 2026-01-11 00:00:00 | Not specified |       61 |      41 |                    1.2 |             0 |                18.9 | 2026-02-05 18:11:24+03:00 |
|  4 | 2026-01-11 00:00:00 | female        |       40 |   

### Дедублицикация (опционально) ↓

In [9]:
client.query_df(f'''
    OPTIMIZE TABLE {database}.YM_local ON CLUSTER '{{cluster}}'
''')

,host,port,status,error,num_hosts_remaining,num_hosts_active
0,clickhouse01,9000,0,,3,0
1,clickhouse02,9000,0,,2,0
2,clickhouse04,9000,0,,1,0
3,clickhouse03,9000,0,,0,0


In [34]:
client.query_df(f'''
    SELECT * FROM {database}.YM_dist
''')

,date,gender,visits,users,avgdaysbetweenvisits,robotvisits,blockedpercentage,updated_at
0,2026-01-10,Not specified,68.0,45.0,2.3,0.0,27.9,2026-02-05 18:11:24+03:00
1,2026-01-10,female,41.0,27.0,2.9,1.0,30.7,2026-02-05 18:11:24+03:00
2,2026-01-10,male,215.0,125.0,2.3,1.0,47.6,2026-02-05 18:11:24+03:00
3,2026-01-11,Not specified,61.0,41.0,1.2,0.0,18.9,2026-02-05 18:11:24+03:00
4,2026-01-11,female,40.0,27.0,3.4,1.0,46.1,2026-02-05 18:11:24+03:00
5,2026-01-11,male,231.0,132.0,1.8,0.0,55.8,2026-02-05 18:11:24+03:00
6,2026-01-12,Not specified,87.0,69.0,3.1,3.0,31.9,2026-02-05 18:11:24+03:00
7,2026-01-12,female,64.0,38.0,3.9,0.0,37.5,2026-02-05 18:11:24+03:00
8,2026-01-12,male,281.0,176.0,3.1,0.0,49.1,2026-02-05 18:11:24+03:00
9,2026-01-18,Not specified,63.0,51.0,2.7,7.0,29.0,2026-02-05 18:11:24+03:00


# из Spark ↓

In [ ]:
import os
from pyspark.sql import SparkSession


schema = 'chatov'

spark = SparkSession.builder \
    .appName("SparkExample") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()


hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")


In [32]:
def pg_spark_click_load(spark, tb_name, tb_name_dist):

	jdbc_url = "jdbc:postgresql://postgres_source:5432/dev"
	db_user = os.getenv("POSTGRES_USER")
	db_password = os.getenv("POSTGRES_PASSWORD")

	tmp_data = (
		spark
		.read 
		.format("jdbc")
		.option("url", jdbc_url)
		.option("user", db_user)
		.option("password", db_password)
		.option("dbtable", tb_name)
		.option("fetchsize", 1000)
		.option("driver", "org.postgresql.Driver")
		.load()
	)

	wrt_data = (
		tmp_data
		.write
		.format("jdbc")
		.option("url", "jdbc:clickhouse://clickhouse01:8123/snccnmb")
		.option("dbtable", "YM_dist")
		.option("user", os.getenv('CLICKHOUSE_USER'))
		.option("password", os.getenv('CLICKHOUSE_PASSWORD'))
		.option("driver", "com.clickhouse.jdbc.ClickHouseDriver")
		.mode("append")
		.save()
	)

pg_spark_click_load(spark, f"{schema}.YM_local", f"{schema}.YM_dist")

26/02/08 09:42:21 WARN ClickHouseConnectionImpl: [JDBC Compliant Mode] Transaction is not supported. Change jdbcCompliant to false to throw SQLException instead.
26/02/08 09:42:21 WARN ClickHouseConnectionImpl: [JDBC Compliant Mode] Transaction is not supported. Change jdbcCompliant to false to throw SQLException instead.
26/02/08 09:42:21 WARN ClickHouseConnectionImpl: [JDBC Compliant Mode] Transaction [829aa4ef-17af-43b2-93f2-7f6155e6204b](2 queries & 0 savepoints) is committed.
26/02/08 09:42:21 WARN ClickHouseConnectionImpl: [JDBC Compliant Mode] Transaction [f0e69313-7dea-4dc9-b7fa-9448fb369710](0 queries & 0 savepoints) is committed.


In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.kafka_out_message;
''')

client.command(f'''
    CREATE TABLE {database}.kafka_out_message
    (
        id UInt32
    )
    ENGINE = Kafka
    SETTINGS
        kafka_broker_list = 'kafka:29092',
        kafka_topic_list = '{database}.clickhouse.topic', -- создай такой же топик
        kafka_group_name = 'clickhouse_consumer_group',
        kafka_format = 'JSONEachRow';
''')

In [14]:
client.command(f'''
    INSERT INTO {database}.kafka_out_message
    SELECT number FROM numbers(30);
''')

In [11]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.data_from_rabbit;
''')

client.command(f'''
   create table {database}.data_from_rabbit 
      (
         json_kafka String
      )
   Engine = Kafka
   SETTINGS
        kafka_broker_list = 'kafka:29092',
        kafka_topic_list = 'source.public.dbz_heartbeat',
        kafka_group_name = 'clickhouse_consumer_group',
        kafka_format = 'JSONAsString'; 
''')

client.command(f'''
    DROP TABLE IF EXISTS {database}.data_rabbit_to_kafka;
''')

client.command(f'''
   create table {database}.data_rabbit_to_kafka 
   (
      before String,
      after String
   )
   Engine = Kafka
   SETTINGS
        kafka_broker_list = 'kafka:29092',
        kafka_topic_list = 'snccnmb.source.data_from_rabbit',
        kafka_group_name = 'clickhouse_consumer_group',
        kafka_format = 'JSONEachRow';
''')


In [12]:
client.command(f'''
    DROP VIEW IF EXISTS {database}.data_from_rabbit_mv;
''')

client.command(f'''
   CREATE MATERIALIZED VIEW {database}.data_from_rabbit_mv TO {database}.data_rabbit_to_kafka AS 
      SELECT 
         simpleJSONExtractRaw(json_kafka, 'before') as before,
         simpleJSONExtractRaw(json_kafka, 'after') as after
         -- для примера тут приведено всего 2 строки, но надеюсь принцип вы поняли
      from {database}.data_from_rabbit ;
''')

# 10 ↓

In [ ]:
client.query_df(f'''
	SELECT 
		event_name,
		COUNT(user_id) as login_count
	FROM user_login t1
	GLOBAL JOIN event t2 ON t1.login_time BETWEEN t2.start_time AND t2.end_time
	GROUP BY event_name
	ORDER BY login_count DESC
''')